# Week 9 Homework

***Due (pushed to your GitHub branch) on 11/13 by 11:59 pm***

## Count QC

Use Python to filter your counts:

1. Remove any gene that has a total expression count (across all samples) of < 10.
2. Remove any gene that was expressed in <= 6 samples.

In [2]:
import pandas as pd

counts_df = pd.read_csv("../8_counting/cow_star_counts.tsv", sep="\t", header=1)
counts_df.iloc[:, 6:32] = counts_df.iloc[:, 6:32].astype('int')
counts_df
star_cols = [c for c in counts_df.columns if "cow_star.bam" in c]
counts_subset = counts_df[["Geneid"] + star_cols].copy()
prefix = "../8_counting/cow_star.bam:"
def strip_prefix(name: str) -> str:
    return name.replace(prefix, "")
counts_subset.rename(columns={c: strip_prefix(c) for c in star_cols}, inplace=True)
counts_subset
sample_cols = [c for c in counts_subset.columns if c != "Geneid"]
counts_numeric = counts_subset[sample_cols].apply(pd.to_numeric, errors="coerce").fillna(0)
counts_subset["total_counts"] = counts_numeric.sum(axis=1)

counts_subset 
counts_summary = counts_subset.loc[counts_subset["total_counts"] >= 10].copy()
counts_summary

,Geneid,cow_star.bam:Wagyu1,cow_star.bam:Wagyu2,cow_star.bam:Wagyu3,cow_star.bam:CRS1,cow_star.bam:CRS2,cow_star.bam:CRS3,total_counts
8,RCAN1,1906.0,1715.0,928.0,758.0,474.0,879.0,6660.0
9,LOC132345447,4.0,0.0,0.0,2.0,4.0,2.0,12.0
14,SMIM11,395.0,472.0,324.0,515.0,502.0,388.0,2596.0
17,MRPS6,316.0,198.0,204.0,266.0,152.0,186.0,1322.0
18,SLC5A3,25.0,28.0,14.0,22.0,28.0,56.0,173.0
...,...,...,...,...,...,...,...,...
32632,KEH36_p02,1613.0,2430.0,2452.0,2030.0,1790.0,1421.0,11736.0
32633,KEH36_t20,2.0,0.0,48.0,31.0,8.0,23.0,112.0
32634,KEH36_p01,8360.0,14413.0,13019.0,12835.0,12569.0,28906.0,90102.0
32635,KEH36_t21,35.0,82.0,44.0,34.0,13.0,19.0,227.0


## Sample QC with clustering and PCA

Use Python to cluster your samples to look for outliers:

1. Generate a heatmap of Euclidean distances.
2. Generate a PCA plot on log<sub>10</sub> + 1 transformed counts

In [9]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.clustermap(
    dists_df,
    cmap="viridis",
    method="average",
    z_score=None,
    standard_scale=None,
    figsize=(8, 8)
)
from sklearn.preprocessing import StandardScaler

# log transform and transpose to samples x genes again
X = np.log10(counts_m.to_numpy() + 1.0).T 
sample_names = counts_m.columns.tolist()

# scale features (genes) across samples
scaler = StandardScaler(with_mean=True, with_std=True)
X_scaled = scaler.fit_transform(X)
X_scaled
sample_ids = list(counts_m.columns)

# create a DataFrame with sample_id
metadata = pd.DataFrame({"sample_id": sample_ids})
metadata
parts = metadata["sample_id"].str.split("_", expand=True)
parts = metadata["sample_id"].str.extract(
    r"^(?P<treatment>[^_]+)_(?P<bio_rep>\d+)(?P<tech_rep>[a-z]+)$"
)
parts

metadata = pd.concat([metadata, parts], axis=1)
metadata = metadata.set_index("sample_id")
metadata

from sklearn.decomposition import PCA

pca = PCA(n_components=10, random_state=0)  
scores = pca.fit_transform(X_scaled)        
# explained = pca.explained_variance_ratio_

scores_df = pd.DataFrame(
    scores[:, :2],  # PC1 and PC2 for plotting
    index=sample_names,
    columns=["PC1", "PC2"]
)

plot_df = scores_df.join(metadata[["treatment", "bio_rep"]], how="left")

# plot
plt.figure(figsize=(7, 6))
sns.scatterplot(
    data=plot_df,
    x="PC1",
    y="PC2",
    hue="treatment",     # color by treatment
    style="bio_rep",     # shape bi bio_rep
    s=80
)
plt.title("PCA: PC1 vs PC2")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", borderaxespad=0.)
plt.tight_layout()
plt.show()

NameError: name 'dists_df' is not defined

## Perform differential expression

Use Python and DESeq2 to perform differential expression analysis on your counts. Generate a volcano plot of the data.

Choose one gene from the paper that was highlighted as being differentially expressed. Compare your data for this gene (raw counts if available, log<sub>2</sub> fold-change, and adjusted p-values) to the data reported in the paper.

Re-generate the PCA plot, now on counts that have had the variance-stabilizing tranformation performed on them.

## Snakemake Pipeline (optional)

1. Copy your `Snakefile` and `config.yaml` from `8_counting` to `9_differential_expression`.  
2. In the `Snakefile`, add the following rules:  
    a. `differential_expression` - write a Python script that performs QC (clustering and PCA) and differential expression. The rule should run the script, and the output should be plots (heatmap, PCA, volcano plot) and a CSV of the DEGs.

This `Snakefile` should not use any `wrappers` because we already have the relevant software installed in the `biol343` conda environment. The pipeline should run to completion when any instructor or classmate runs `snakemake --use-conda`.